# Dataset Validation & Analysis

**Purpose**: Validate dataset structure and provide detailed statistics.

**Prerequisites**: Run `01_dataset_pipeline_local.ipynb` first


## Section 1: Imports & Load Data


In [69]:
import os
import json
import random
from pathlib import Path
from collections import Counter

In [70]:
# Configure paths (works in both Colab and local)
try:
    from google.colab import drive
    # Running in Colab - use Google Drive
    drive.mount('/content/drive')
    BASE_DIR = Path("/content/drive/MyDrive/CSI_Project")
    IS_COLAB = True
    print("✓ Running in Colab - using Google Drive paths")
except ImportError:
    # Running locally
    BASE_DIR = Path("/Users/anas/Projects/code-security-identifier")
    IS_COLAB = False
    print("✓ Running locally - using local paths")

DATASETS_DIR = BASE_DIR / "datasets"


def read_jsonl(path):
    """
    Read JSONL file into list of records.
    """
    if not os.path.exists(path):
        return []
    with open(path) as f:
        return [json.loads(line) for line in f]


# Load datasets
train = read_jsonl(DATASETS_DIR / "FINAL_train.jsonl")
val = read_jsonl(DATASETS_DIR / "FINAL_val.jsonl")

print(f"✓ Train: {len(train):,} functions")
print(f"✓ Val:   {len(val):,} functions")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Running in Colab - using Google Drive paths
✓ Train: 3,677 functions
✓ Val:   408 functions


## Section 2: Structure Validation


In [71]:
print("Validating dataset structure...\n")

issues = []

for split_name, data in [("train", train), ("val", val)]:
    for i, r in enumerate(data):
        # Check required fields
        required = ["lines", "raw_lines", "label", "type", "cwe_id", "dataset_source"]
        for field in required:
            if field not in r:
                issues.append(f"{split_name}[{i}]: missing '{field}'")

        # Check mismatches
        n_labels = len(r.get("label", []))
        n_lines = len(r.get("lines", []))
        n_raw = len(r.get("raw_lines", []))
        n_types = len(r.get("type", []))

        if n_labels != n_lines:
            issues.append(f"{split_name}[{i}]: label({n_labels}) != lines({n_lines})")
        if n_labels != n_types:
            issues.append(f"{split_name}[{i}]: label({n_labels}) != type({n_types})")
        if n_labels != n_raw:
            issues.append(f"{split_name}[{i}]: label({n_labels}) != raw_lines({n_raw})")

        # Check for empty labels
        if n_labels == 0:
            issues.append(f"{split_name}[{i}]: empty label list")

if issues:
    print(f"⚠ Found {len(issues)} structure issues:")
    for issue in issues[:10]:
        print(f"  - {issue}")
    if len(issues) > 10:
        print(f"  ... and {len(issues) - 10} more")
else:
    print("✓ All datasets have valid structure")
    print(f"✓ All {len(train) + len(val):,} records have correct format")

Validating dataset structure...

✓ All datasets have valid structure
✓ All 4,085 records have correct format


## Section 3: Distribution Summary


In [72]:
print("\n" + "=" * 70)
print("DATASET SUMMARY")
print("=" * 70)

# Overview
print(f"\nSplit sizes (functions):")
print(f"  Train: {len(train):,}")
print(f"  Val:   {len(val):,}")
print(f"  Total: {len(train) + len(val):,}")

# Vulnerability distribution (functions)
print(f"\nVulnerability distribution (functions):")
for name, data in [("Train", train), ("Val", val)]:
    total = len(data)
    vuln = sum(1 for r in data if sum(r["label"]) > 0)
    safe = total - vuln
    pct = vuln / total * 100 if total else 0
    print(
        f"  {name:>5}: {vuln:>7,} vulnerable / {safe:>7,} safe ({pct:>5.1f}% vulnerable)"
    )

# Statement-level distribution
print(f"\nVulnerability distribution (statements):")
for name, data in [("Train", train), ("Val", val)]:
    total_s = sum(len(r["label"]) for r in data)
    vuln_s = sum(sum(r["label"]) for r in data)
    safe_s = total_s - vuln_s
    pct = vuln_s / total_s * 100 if total_s else 0
    print(
        f"  {name:>5}: {vuln_s:>9,} vulnerable / {safe_s:>9,} safe ({pct:>5.1f}% vulnerable)"
    )


DATASET SUMMARY

Split sizes (functions):
  Train: 3,677
  Val:   408
  Total: 4,085

Vulnerability distribution (functions):
  Train:   2,214 vulnerable /   1,463 safe ( 60.2% vulnerable)
    Val:     260 vulnerable /     148 safe ( 63.7% vulnerable)

Vulnerability distribution (statements):
  Train:    22,339 vulnerable /   234,971 safe (  8.7% vulnerable)
    Val:     2,842 vulnerable /    28,439 safe (  9.1% vulnerable)


## Section 4: Dataset Sources


In [73]:
print(f"\nDataset sources in training set:")
sources = Counter(r.get("dataset_source", "?") for r in train)
for source, count in sources.most_common():
    pct = count / len(train) * 100
    print(f"  {source:>15}: {count:>7,} ({pct:>5.1f}%)")

print(f"\nDataset sources in validation set:")
sources_val = Counter(r.get("dataset_source", "?") for r in val)
for source, count in sources_val.most_common():
    pct = count / len(val) * 100
    print(f"  {source:>15}: {count:>7,} ({pct:>5.1f}%)")


Dataset sources in training set:
        funclevel:   2,452 ( 66.7%)
           vudenc:   1,117 ( 30.4%)
     securityeval:     108 (  2.9%)

Dataset sources in validation set:
        funclevel:     266 ( 65.2%)
           vudenc:     129 ( 31.6%)
     securityeval:      13 (  3.2%)


## Section 5: CWE Distribution


In [74]:
print(f"\nCWE types in training set (vulnerable functions only):")
cwes = Counter(r.get("cwe_id", "?") for r in train if sum(r["label"]) > 0)
for cwe, count in cwes.most_common():
    print(f"  {cwe:>12}: {count:>6,}")

print(f"\nCWE types in validation set (vulnerable functions only):")
cwes_val = Counter(r.get("cwe_id", "?") for r in val if sum(r["label"]) > 0)
for cwe, count in cwes_val.most_common():
    print(f"  {cwe:>12}: {count:>6,}")


CWE types in training set (vulnerable functions only):
       unknown:    755
       CWE-089:    487
       CWE-079:    176
       CWE-022:    170
       CWE-352:    162
       CWE-601:    159
       CWE-077:    126
       CWE-094:     81
       CWE-611:      5
       CWE-020:      5
       CWE-502:      4
       CWE-918:      4
       CWE-327:      3
       CWE-730:      3
       CWE-347:      3
       CWE-703:      3
       CWE-117:      3
       CWE-295:      3
       CWE-643:      2
       CWE-326:      2
       CWE-521:      2
       CWE-259:      2
      CWE-1333:      2
       CWE-319:      2
       CWE-400:      2
       CWE-116:      2
       CWE-321:      2
       CWE-434:      2
       CWE-113:      2
       CWE-798:      2
       CWE-080:      1
       CWE-377:      1
       CWE-252:      1
       CWE-641:      1
       CWE-760:      1
       CWE-406:      1
       CWE-269:      1
       CWE-414:      1
       CWE-215:      1
       CWE-090:      1
       CWE-827:      1
 

## Section 6: File Sizes


In [75]:
print(f"\nFile sizes:")
for fname in ["FINAL_train.jsonl", "FINAL_val.jsonl"]:
    path = DATASETS_DIR / fname
    size_mb = os.path.getsize(path) / (1024 * 1024)
    size_lines = sum(1 for _ in open(path))
    print(f"  {fname:>20}: {size_mb:>7.1f} MB ({size_lines:>8,} lines)")

print(f"\n✓ Output directory: {DATASETS_DIR}")


File sizes:
     FINAL_train.jsonl:    25.4 MB (   3,677 lines)
       FINAL_val.jsonl:     3.0 MB (     408 lines)

✓ Output directory: /content/drive/MyDrive/CSI_Project/datasets


## Section 7: Sample Records


In [76]:
print("\nRandom vulnerable samples from training set:\n")

# Pick 3 vulnerable functions at random
random.seed(0)
vuln_samples = [r for r in train if sum(r["label"]) > 0]
picks = random.sample(vuln_samples, min(3, len(vuln_samples)))

for i, sample in enumerate(picks):
    cwe = sample.get("cwe_id", "?")
    src = sample.get("dataset_source", "?")
    print(f"--- Sample {i+1} | {cwe} | from: {src} ---")

    lines = sample.get("raw_lines", sample.get("lines", []))
    labels = sample["label"]

    for j, (line, lbl) in enumerate(zip(lines[:15], labels[:15])):
        mark = ">>" if lbl == 1 else "  "
        print(f"  {mark} {j+1:>3} | {line[:75]}")

    if len(lines) > 15:
        print(f"       ... ({len(lines) - 15} more lines)")
    print()


Random vulnerable samples from training set:

--- Sample 1 | CWE-089 | from: vudenc ---
       1 | import sys
       2 | import logging
       3 | from django.db import connection, DatabaseError
       4 | from reviewus.settings import DEBUG
       5 | logger = logging.getLogger(__name__)
       6 | class DBConnection:
       7 |   instance = None
       8 |   con = None
       9 |   def __new__(cls):
      10 |     if DBConnection.instance is None:
      11 |       DBConnection.instance = object.__new__(cls)
      12 |     return DBConnection.instance
      13 |   def __init__(self):
      14 |     if DBConnection.con is None:
      15 |       try:
       ... (78 more lines)

--- Sample 2 | CWE-022 | from: funclevel ---
       1 | def _extract_tar_file(tar, filename, b_dest, b_temp_path, expected_hash=Non
       2 |     n_filename = to_native(filename, errors='surrogate_or_strict')
       3 |     try:
       4 |         member = tar.getmember(n_filename)
       5 |     except KeyErro

## Section 8: Function Length Distribution


In [77]:
print("Train split - statements per function:")
lengths = [len(r["label"]) for r in train]
print(f"  Min:    {min(lengths):>6}")
print(f"  Max:    {max(lengths):>6}")
print(f"  Mean:   {sum(lengths)/len(lengths):>6.1f}")
print(f"  Median: {sorted(lengths)[len(lengths)//2]:>6}")

# Percentiles
over_50 = sum(1 for l in lengths if l > 50)
over_100 = sum(1 for l in lengths if l > 100)
over_200 = sum(1 for l in lengths if l > 200)
print(f"\n  >50 statements:   {over_50:>6,} ({over_50/len(lengths)*100:>5.1f}%)")
print(f"  >100 statements:  {over_100:>6,} ({over_100/len(lengths)*100:>5.1f}%)")
print(f"  >200 statements:  {over_200:>6,} ({over_200/len(lengths)*100:>5.1f}%)")

Train split - statements per function:
  Min:         1
  Max:       771
  Mean:     70.0
  Median:     31

  >50 statements:    1,316 ( 35.8%)
  >100 statements:     725 ( 19.7%)
  >200 statements:     309 (  8.4%)


## Section 9: Vulnerable Lines per Function


In [78]:
print("\nTrain split - vulnerable statements per vulnerable function:")
vuln_counts = [sum(r["label"]) for r in train if sum(r["label"]) > 0]

if vuln_counts:
    print(f"  Min:    {min(vuln_counts):>6}")
    print(f"  Max:    {max(vuln_counts):>6}")
    print(f"  Mean:   {sum(vuln_counts)/len(vuln_counts):>6.1f}")
    print(f"  Median: {sorted(vuln_counts)[len(vuln_counts)//2]:>6}")
else:
    print("  No vulnerable functions found")


Train split - vulnerable statements per vulnerable function:
  Min:         1
  Max:       543
  Mean:     10.1
  Median:      3


## Section 10: Final Data Quality Gate

This section enforces quality thresholds before model training.

In [79]:
import json
import hashlib


def record_signature(record):
    payload = {
        "lines": record.get("lines", []),
        "label": record.get("label", []),
        "cwe_id": record.get("cwe_id", "unknown"),
        "dataset_source": record.get("dataset_source", "unknown"),
    }
    text = json.dumps(payload, sort_keys=True)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def split_metrics(name, data):
    if not data:
        return {
            "name": name,
            "records": 0,
            "function_vulnerable_pct": 0.0,
            "statement_vulnerable_pct": 0.0,
            "duplicate_pct": 0.0,
            "unknown_cwe_pct": 0.0,
            "all_vulnerable_long_functions_pct": 0.0,
        }

    records = len(data)
    vulnerable_functions = sum(1 for r in data if sum(r.get("label", [])) > 0)

    total_statements = sum(len(r.get("label", [])) for r in data)
    vulnerable_statements = sum(sum(r.get("label", [])) for r in data)

    signatures = [record_signature(r) for r in data]
    unique_signatures = len(set(signatures))
    duplicate_count = records - unique_signatures

    unknown_cwe = sum(
        1 for r in data if str(r.get("cwe_id", "")).lower() in {"", "unknown", "?"}
    )
    all_vuln_long = sum(
        1
        for r in data
        if len(r.get("label", [])) > 10
        and sum(r.get("label", [])) == len(r.get("label", []))
    )

    return {
        "name": name,
        "records": records,
        "function_vulnerable_pct": vulnerable_functions / records * 100,
        "statement_vulnerable_pct": (
            (vulnerable_statements / total_statements * 100)
            if total_statements > 0
            else 0.0
        ),
        "duplicate_pct": duplicate_count / records * 100,
        "unknown_cwe_pct": unknown_cwe / records * 100,
        "all_vulnerable_long_functions_pct": all_vuln_long / records * 100,
    }


test = read_jsonl(DATASETS_DIR / "FINAL_test.jsonl")
splits = [("train", train), ("val", val)]
if test:
    splits.append(("test", test))

quality = {name: split_metrics(name, data) for name, data in splits}

print("Final quality-gate metrics:\n")
for name, metrics in quality.items():
    print(f"[{name}]")
    print(f"  records: {metrics['records']:,}")
    print(f"  vulnerable functions: {metrics['function_vulnerable_pct']:.2f}%")
    print(f"  vulnerable statements: {metrics['statement_vulnerable_pct']:.2f}%")
    print(f"  duplicates: {metrics['duplicate_pct']:.2f}%")
    print(f"  unknown CWE: {metrics['unknown_cwe_pct']:.2f}%")
    print(
        f"  all-vulnerable long functions: {metrics['all_vulnerable_long_functions_pct']:.2f}%"
    )
    print()

critical_failures = []
warnings = []

for name, metrics in quality.items():
    if metrics["records"] == 0:
        critical_failures.append(f"{name}: empty split")
        continue

    if not (3 <= metrics["function_vulnerable_pct"] <= 80):
        critical_failures.append(
            f"{name}: vulnerable function ratio out of expected range ({metrics['function_vulnerable_pct']:.2f}%)"
        )

    if not (1 <= metrics["statement_vulnerable_pct"] <= 40):
        critical_failures.append(
            f"{name}: vulnerable statement ratio out of expected range ({metrics['statement_vulnerable_pct']:.2f}%)"
        )

    if metrics["duplicate_pct"] > 15:
        critical_failures.append(
            f"{name}: duplicate ratio too high ({metrics['duplicate_pct']:.2f}%)"
        )

    if metrics["unknown_cwe_pct"] > 35:
        warnings.append(
            f"{name}: unknown CWE ratio is high ({metrics['unknown_cwe_pct']:.2f}%)"
        )

    if metrics["all_vulnerable_long_functions_pct"] > 5:
        warnings.append(
            f"{name}: high share of long all-vulnerable functions ({metrics['all_vulnerable_long_functions_pct']:.2f}%)"
        )

report = {
    "quality": quality,
    "critical_failures": critical_failures,
    "warnings": warnings,
    "status": "PASS" if not critical_failures else "FAIL",
}

report_path = DATASETS_DIR / "validation_quality_gate.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print("=" * 70)
print(f"Quality gate report saved to: {report_path}")
print(f"Status: {report['status']}")
if warnings:
    print("Warnings:")
    for w in warnings:
        print(f"  - {w}")

if critical_failures:
    print("Critical failures:")
    for fail in critical_failures:
        print(f"  - {fail}")
    raise AssertionError("Data quality gate failed. Resolve issues before training.")
else:
    print("PASS: Dataset quality is acceptable for training.")
print("=" * 70)

Final quality-gate metrics:

[train]
  records: 3,677
  vulnerable functions: 60.21%
  vulnerable statements: 8.68%
  duplicates: 0.00%
  unknown CWE: 49.99%
  all-vulnerable long functions: 2.42%

[val]
  records: 408
  vulnerable functions: 63.73%
  vulnerable statements: 9.09%
  duplicates: 0.00%
  unknown CWE: 48.53%
  all-vulnerable long functions: 2.45%

Quality gate report saved to: /content/drive/MyDrive/CSI_Project/datasets/validation_quality_gate.json
Status: PASS
Warnings:
  - train: unknown CWE ratio is high (49.99%)
  - val: unknown CWE ratio is high (48.53%)
PASS: Dataset quality is acceptable for training.
